# EfficientNet-B2 Signature Classification (Genuine vs. Forged)

**Workflow**
1. **Digital Image Processing (DIP)**: Otsu binarisation, contrast stretching, median filter, high-boost filter, Sobel gradient.
2. **Part A - Training (run once)**: fine-tune a pretrained **EfficientNet-B2**; the best model is **saved** to `models/efficientnet_b2_signature.pt`.
3. **Part B - Testing from the saved model**: load the checkpoint and evaluate on the untouched test set - **no training needed**.
4. **Novelty and modifications** of EfficientNet-B2 for this signature-forgery case study (last section).

> **Presenting?** Skip Part A. Jump straight to **PART B** (below the training section) and run from its first cell.
> Part B is self-contained: it works right after a kernel restart.

**Fixed version notes:** the live demo now reports both the saved validation-calibrated threshold decision and the 0.50-threshold decision; invalid image paths raise an error rather than silently substituting a blank image; checkpoint class metadata is checked when available. These fixes improve diagnostics but do not guarantee correct predictions on signatures from writers absent from CEDAR.

---
# PART A - TRAINING 
## 1. Setup
`FORCE_RETRAIN = False` means: if a saved model already exists, training is skipped. Set it to `True` to train again.

In [ ]:
import os
import glob
import json
import zipfile
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score, confusion_matrix, classification_report,
                             matthews_corrcoef, cohen_kappa_score)

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Compute device
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Using compute device: {device}')

# Paths used by BOTH training (Part A) and testing (Part B)
CKPT_PATH  = 'models/efficientnet_b2_signature.pt'   # saved model
SPLIT_PATH = 'results/split.json'                    # exact train/val/test file lists
CLASS_NAMES = ['Genuine', 'Forged']                  # label 0 / label 1
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

FORCE_RETRAIN = False   # True -> always retrain and overwrite the saved model

## 2. Dataset Setup
Extracts any dataset zip found, or uses `data/` as is.

In [ ]:
# Automatic dataset extraction if zip exists
zip_candidates = glob.glob('*.zip') + glob.glob('data/*.zip')
if zip_candidates and not (os.path.exists('data/train') and len(glob.glob('data/train/*/*')) > 0):
    print(f'Extracting {zip_candidates[0]} into data/ ...')
    with zipfile.ZipFile(zip_candidates[0], 'r') as z:
        z.extractall('data')
print('Dataset ready for training/testing.')

## 3. Digital Image Processing (DIP) Pipeline
Sequential DIP stages. Their outputs become the **3 input channels** of EfficientNet-B2.

In [ ]:
OUT_H, OUT_W = 192, 384
MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)

def ink_mask(gray, min_area=12):
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    _, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    keep = np.zeros(n, bool)
    keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= min_area
    return keep[labels]

def canonicalise(gray, binarize=True, margin=8):
    mask = ink_mask(gray)
    if not mask.any():
        return np.full_like(gray, 255)
    ys, xs = np.where(mask)
    y0, y1 = max(ys.min() - margin, 0), min(ys.max() + margin + 1, gray.shape[0])
    x0, x1 = max(xs.min() - margin, 0), min(xs.max() + margin + 1, gray.shape[1])
    g, m = gray[y0:y1, x0:x1], mask[y0:y1, x0:x1]
    out = np.full(g.shape, 255, np.uint8)
    if binarize:
        out[m] = 0
    else:
        ink = g[m].astype(np.float64)
        lo, hi = np.percentile(ink, 2), np.percentile(ink, 98)
        scaled = (ink - lo) / max(hi - lo, 1.0) * 160.0
        out[m] = np.clip(scaled, 0, 160).astype(np.uint8)
    return out

def letterbox(img, h=OUT_H, w=OUT_W, fill=255):
    s = min(h / img.shape[0], w / img.shape[1])
    nh, nw = max(int(round(img.shape[0] * s)), 1), max(int(round(img.shape[1] * s)), 1)
    r = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    out = np.full((h, w), fill, np.uint8)
    y, x = (h - nh) // 2, (w - nw) // 2
    out[y:y + nh, x:x + nw] = r
    return out

def contrast_stretching(img):
    r_min, r_max = float(np.min(img)), float(np.max(img))
    if r_max == r_min:
        return img.copy()
    stretched = (img.astype(np.float64) - r_min) * (255.0 / (r_max - r_min))
    return np.uint8(np.clip(stretched, 0, 255))

def median_filter(img, ksize=3):
    return cv2.medianBlur(img, ksize)

def high_boost_filter(img, ksize=3, k=1.5):
    blurred = cv2.blur(img, (ksize, ksize)).astype(np.float64)
    mask = img.astype(np.float64) - blurred
    boosted = img.astype(np.float64) + k * mask
    return np.uint8(np.clip(boosted, 0, 255))

def sobel_edge_detection(img, ksize=3):
    gx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=ksize)
    gy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=ksize)
    return np.uint8(np.clip(np.sqrt(gx**2 + gy**2), 0, 255))

def stages_to_channels(canon):
    s1 = contrast_stretching(canon)
    s2 = median_filter(s1, ksize=3)
    s3 = high_boost_filter(s2, ksize=3, k=1.5)
    s4 = sobel_edge_detection(s3, ksize=3)
    return np.stack([s2, s3, s4], axis=0)      # 3 DIP-derived channels -> EfficientNet input

def augment_signature(canon, rng):
    h, w = canon.shape
    ang, sc = rng.uniform(-6, 6), rng.uniform(0.82, 1.0)
    M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, sc)
    M[0, 1] += np.tan(np.radians(rng.uniform(-8, 8)))
    M[:, 2] += (rng.uniform(-10, 10), rng.uniform(-8, 8))
    img = cv2.warpAffine(canon, M, (w, h), flags=cv2.INTER_LINEAR, borderValue=255)
    r = rng.random()
    if r < 0.25:
        img = cv2.erode(img, cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3)))
    elif r < 0.5:
        img = cv2.dilate(img, cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3)))
    return np.where(img < 128, 0, 255).astype(np.uint8)

## 4. Dataset Loader and Train / Validation / Test Split
The split is **saved to `results/split.json`**, so Part B tests on exactly the same unseen test images.
Validation (15 % of the training portion) is used to choose the best epoch and the decision threshold; **the test set is never touched during training**.

In [ ]:
def load_canonical(path):
    """Read an image and apply the canonicalisation step (crop + binarise + letterbox)."""
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise ValueError(f"Could not read image: {path}. Check that the path is correct and the file is a supported image.")
    if gray.size == 0:
        raise ValueError(f"Image is empty: {path}")
    return letterbox(canonicalise(gray, binarize=True))

class SignatureDataset(Dataset):
    def __init__(self, file_paths, labels, is_train=False, seed=None):
        self.file_paths = list(file_paths)
        self.labels = np.array(labels, dtype=np.float32)
        self.is_train = is_train
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        canon = load_canonical(self.file_paths[idx])
        if self.is_train:
            canon = augment_signature(canon, self.rng)
        channels = stages_to_channels(canon)
        x = (channels.astype(np.float32) / 255.0 - MEAN) / STD
        return torch.from_numpy(x), torch.tensor(self.labels[idx])

def gather_data(data_root='data'):
    """Returns (train_paths, test_paths, train_labels, test_labels). Label 0 = genuine, 1 = forged."""
    train_g = sorted(glob.glob(os.path.join(data_root, 'train', 'genuine', '*.*')))
    train_f = sorted(glob.glob(os.path.join(data_root, 'train', 'forged', '*.*')))
    test_g = sorted(glob.glob(os.path.join(data_root, 'test', 'genuine', '*.*')))
    test_f = sorted(glob.glob(os.path.join(data_root, 'test', 'forged', '*.*')))

    if train_g or train_f:
        tr_p = train_g + train_f
        tr_l = [0] * len(train_g) + [1] * len(train_f)
        te_p = test_g + test_f
        te_l = [0] * len(test_g) + [1] * len(test_f)
        return tr_p, te_p, tr_l, te_l

    all_files = sorted(glob.glob(os.path.join(data_root, '**', '*.*'), recursive=True))
    img_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
    paths, labels = [], []
    for f in all_files:
        ext = os.path.splitext(f)[1].lower()
        if ext in img_exts:
            fn = os.path.basename(f).lower()
            parent = os.path.basename(os.path.dirname(f)).lower()
            if 'original' in fn or 'genuine' in fn or 'original' in parent or 'genuine' in parent:
                paths.append(f)
                labels.append(0)
            elif 'forg' in fn or 'forg' in parent:
                paths.append(f)
                labels.append(1)

    if len(paths) > 0:
        return train_test_split(paths, labels, test_size=0.2, random_state=SEED, stratify=labels)
    return [], [], [], []

def make_splits(data_root='data', val_frac=0.15):
    """Train / validation / test. Validation is carved out of the training portion, so the
    test set is never used for choosing the best epoch or the decision threshold."""
    tr_p, te_p, tr_l, te_l = gather_data(data_root)
    if len(tr_p) == 0:
        return None
    tr_p, va_p, tr_l, va_l = train_test_split(tr_p, tr_l, test_size=val_frac,
                                              random_state=SEED, stratify=tr_l)
    return {'train': (tr_p, tr_l), 'val': (va_p, va_l), 'test': (te_p, te_l)}

def save_splits(splits, path=SPLIT_PATH):
    out = {k: {'paths': [str(p) for p in v[0]], 'labels': [int(l) for l in v[1]]} for k, v in splits.items()}
    with open(path, 'w') as f:
        json.dump(out, f, indent=1)

def load_splits(path=SPLIT_PATH):
    with open(path) as f:
        d = json.load(f)
    return {k: (v['paths'], v['labels']) for k, v in d.items()}

if os.path.exists(SPLIT_PATH) and not FORCE_RETRAIN:
    splits = load_splits()
    print(f'Loaded existing split from {SPLIT_PATH}')
else:
    splits = make_splits('data', val_frac=0.15)
    if splits is not None:
        save_splits(splits)
        print(f'Saved split to {SPLIT_PATH}')

if splits is None:
    print('No images found in data/ yet.')
else:
    for name in ['train', 'val', 'test']:
        lab = np.array(splits[name][1])
        print(f'{name:>5}: {len(lab):5d} images  (genuine={int((lab == 0).sum())}, forged={int((lab == 1).sum())})')

## 5. EfficientNet-B2 Model
Pretrained `efficientnet_b2` with its ImageNet head replaced by a **single linear output** (forged logit, trained with BCE-with-logits).

In [ ]:
def build_model(pretrained=True, dropout=0.5):
    weights = EfficientNet_B2_Weights.DEFAULT if pretrained else None
    m = efficientnet_b2(weights=weights)
    m.classifier[0].p = dropout                                   # stronger dropout (default 0.3)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)   # 1000-class head -> 1 forged logit
    return m

model = build_model(pretrained=True).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'EfficientNet-B2 instantiated successfully. Parameters: {n_params/1e6:.2f} M')

## 6. Training and Saving the Model
Each epoch is scored on the **validation** split. The best epoch (highest validation AUC) is **saved** together with its
validation-calibrated decision threshold (the EER point) and the preprocessing configuration.

In [ ]:
def train(model, splits, epochs=10, batch_size=16, lr=1e-4):
    tr_p, tr_l = splits['train']
    va_p, va_l = splits['val']
    train_ds = SignatureDataset(tr_p, tr_l, is_train=True, seed=SEED)
    val_ds = SignatureDataset(va_p, va_l, is_train=False)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    head = list(model.classifier.parameters())
    head_ids = {id(p) for p in head}
    body = [p for p in model.parameters() if id(p) not in head_ids]
    optimizer = torch.optim.AdamW([{'params': body, 'lr': lr}, {'params': head, 'lr': lr * 10}], weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.BCEWithLogitsLoss()

    best_auc, best_thr, history = -1.0, 0.5, []

    for ep in range(1, epochs + 1):
        # ---- train ----
        model.train()
        running_loss, seen = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            smoothed_y = yb * 0.9 + 0.05            # label smoothing
            optimizer.zero_grad()
            loss = criterion(model(xb).squeeze(1), smoothed_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(yb)
            seen += len(yb)
        scheduler.step()
        running_loss /= max(seen, 1)

        # ---- validate ----
        model.eval()
        v_logits, v_targets = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                v_logits.append(model(xb.to(device)).squeeze(1).cpu().numpy())
                v_targets.append(yb.numpy())
        logits = np.concatenate(v_logits)
        targets = np.concatenate(v_targets).astype(int)
        probs = 1.0 / (1.0 + np.exp(-logits))
        p = np.clip(probs, 1e-7, 1 - 1e-7)
        val_loss = float(-np.mean(targets * np.log(p) + (1 - targets) * np.log(1 - p)))
        acc = accuracy_score(targets, (probs >= 0.5).astype(int))
        auc = roc_auc_score(targets, probs) if len(np.unique(targets)) > 1 else 0.5

        # EER operating point on the validation set
        fpr, tpr, thr_arr = roc_curve(targets, probs)
        eer_idx = int(np.nanargmin(np.abs((1 - tpr) - fpr)))
        ep_thr = float(thr_arr[eer_idx]) if np.isfinite(thr_arr[eer_idx]) else 0.5

        print(f'Epoch [{ep:02d}/{epochs:02d}] Train Loss: {running_loss:.4f} | Val Loss: {val_loss:.4f} | '
              f'Val Acc: {acc:.4f} | Val AUC: {auc:.4f}')
        history.append({'epoch': ep, 'train_loss': running_loss, 'val_loss': val_loss,
                        'val_acc': float(acc), 'val_auc': float(auc), 'val_eer_threshold': ep_thr})

        if auc > best_auc:
            best_auc, best_thr = float(auc), ep_thr
            torch.save({
                'model': model.state_dict(),
                'prob_thr': best_thr,                       # decision threshold (probability of "Forged")
                'val_auc': best_auc,
                'val_acc': float(acc),
                'best_epoch': ep,
                'history': history,
                'config': {'arch': 'efficientnet_b2', 'dropout': float(model.classifier[0].p),
                           'img_h': OUT_H, 'img_w': OUT_W,
                           'mean': MEAN.flatten().tolist(), 'std': STD.flatten().tolist(),
                           'class_names': CLASS_NAMES, 'seed': SEED},
            }, CKPT_PATH)
            print(f'  --> Best so far. Saved {CKPT_PATH}')

    pd.DataFrame(history).to_csv('results/training_history.csv', index=False)
    print('Saved results/training_history.csv')

    # keep the best weights in memory
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    print(f"Best epoch: {ckpt['best_epoch']} | Val AUC: {ckpt['val_auc']:.4f} | Threshold: {ckpt['prob_thr']:.4f}")
    return history, best_thr

if splits is None:
    print('No images found - cannot train.')
elif os.path.exists(CKPT_PATH) and os.path.exists(SPLIT_PATH) and not FORCE_RETRAIN:
    print(f'Saved model found at {CKPT_PATH} - skipping training. (Set FORCE_RETRAIN = True to retrain.)')
else:
    history, calibrated_thr = train(model, splits, epochs=10)

## 7. Training Curves

In [ ]:
if os.path.exists('results/training_history.csv'):
    hist_df = pd.read_csv('results/training_history.csv')
    best_ep = int(hist_df.loc[hist_df['val_auc'].idxmax(), 'epoch'])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(hist_df['epoch'], hist_df['train_loss'], 'o-', color='#DC2626', label='Train Loss')
    if 'val_loss' in hist_df:
        axes[0].plot(hist_df['epoch'], hist_df['val_loss'], 'd--', color='#F59E0B', label='Val Loss')
    axes[0].set_title('Loss', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].grid(True, linestyle='--', alpha=0.6); axes[0].legend()

    axes[1].plot(hist_df['epoch'], hist_df['val_acc'], 's-', color='#2563EB', label='Val Accuracy')
    axes[1].plot(hist_df['epoch'], hist_df['val_auc'], '^-', color='#059669', label='Val AUC')
    axes[1].axvline(best_ep, color='gray', linestyle=':', label=f'Saved model (epoch {best_ep})')
    axes[1].set_title('Validation Performance', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Score')
    axes[1].grid(True, linestyle='--', alpha=0.6); axes[1].legend()

    plt.tight_layout()
    plt.savefig('results/training_curves.png', dpi=300)
    plt.show()
else:
    print('No training history found yet - run the training cell first.')

---
# PART B - TESTING WITH THE SAVED MODEL  


**No training happens in this part.** It loads `models/efficientnet_b2_signature.pt` and evaluates it on the held-out test images listed in `results/split.json`.

Requirements: the `models/` and `results/` folders (and the `data/` folder) next to this notebook.
Works right after a kernel restart - just run the cells below from top to bottom.

In [ ]:
# ---- B1. Standalone setup (imports + pipeline definitions identical to Part A) ----
import os
import glob
import json
import zipfile
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score, confusion_matrix, classification_report,
                             matthews_corrcoef, cohen_kappa_score)

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Compute device
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Using compute device: {device}')

# Paths used by BOTH training (Part A) and testing (Part B)
CKPT_PATH  = 'models/efficientnet_b2_signature.pt'   # saved model
SPLIT_PATH = 'results/split.json'                    # exact train/val/test file lists
CLASS_NAMES = ['Genuine', 'Forged']                  # label 0 / label 1
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

OUT_H, OUT_W = 192, 384
MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)

def ink_mask(gray, min_area=12):
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    _, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    keep = np.zeros(n, bool)
    keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= min_area
    return keep[labels]

def canonicalise(gray, binarize=True, margin=8):
    mask = ink_mask(gray)
    if not mask.any():
        return np.full_like(gray, 255)
    ys, xs = np.where(mask)
    y0, y1 = max(ys.min() - margin, 0), min(ys.max() + margin + 1, gray.shape[0])
    x0, x1 = max(xs.min() - margin, 0), min(xs.max() + margin + 1, gray.shape[1])
    g, m = gray[y0:y1, x0:x1], mask[y0:y1, x0:x1]
    out = np.full(g.shape, 255, np.uint8)
    if binarize:
        out[m] = 0
    else:
        ink = g[m].astype(np.float64)
        lo, hi = np.percentile(ink, 2), np.percentile(ink, 98)
        scaled = (ink - lo) / max(hi - lo, 1.0) * 160.0
        out[m] = np.clip(scaled, 0, 160).astype(np.uint8)
    return out

def letterbox(img, h=OUT_H, w=OUT_W, fill=255):
    s = min(h / img.shape[0], w / img.shape[1])
    nh, nw = max(int(round(img.shape[0] * s)), 1), max(int(round(img.shape[1] * s)), 1)
    r = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    out = np.full((h, w), fill, np.uint8)
    y, x = (h - nh) // 2, (w - nw) // 2
    out[y:y + nh, x:x + nw] = r
    return out

def contrast_stretching(img):
    r_min, r_max = float(np.min(img)), float(np.max(img))
    if r_max == r_min:
        return img.copy()
    stretched = (img.astype(np.float64) - r_min) * (255.0 / (r_max - r_min))
    return np.uint8(np.clip(stretched, 0, 255))

def median_filter(img, ksize=3):
    return cv2.medianBlur(img, ksize)

def high_boost_filter(img, ksize=3, k=1.5):
    blurred = cv2.blur(img, (ksize, ksize)).astype(np.float64)
    mask = img.astype(np.float64) - blurred
    boosted = img.astype(np.float64) + k * mask
    return np.uint8(np.clip(boosted, 0, 255))

def sobel_edge_detection(img, ksize=3):
    gx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=ksize)
    gy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=ksize)
    return np.uint8(np.clip(np.sqrt(gx**2 + gy**2), 0, 255))

def stages_to_channels(canon):
    s1 = contrast_stretching(canon)
    s2 = median_filter(s1, ksize=3)
    s3 = high_boost_filter(s2, ksize=3, k=1.5)
    s4 = sobel_edge_detection(s3, ksize=3)
    return np.stack([s2, s3, s4], axis=0)      # 3 DIP-derived channels -> EfficientNet input

def augment_signature(canon, rng):
    h, w = canon.shape
    ang, sc = rng.uniform(-6, 6), rng.uniform(0.82, 1.0)
    M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, sc)
    M[0, 1] += np.tan(np.radians(rng.uniform(-8, 8)))
    M[:, 2] += (rng.uniform(-10, 10), rng.uniform(-8, 8))
    img = cv2.warpAffine(canon, M, (w, h), flags=cv2.INTER_LINEAR, borderValue=255)
    r = rng.random()
    if r < 0.25:
        img = cv2.erode(img, cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3)))
    elif r < 0.5:
        img = cv2.dilate(img, cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3)))
    return np.where(img < 128, 0, 255).astype(np.uint8)

def load_canonical(path):
    """Read an image and apply the canonicalisation step (crop + binarise + letterbox)."""
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        gray = np.full((OUT_H, OUT_W), 255, dtype=np.uint8)
    return letterbox(canonicalise(gray, binarize=True))

class SignatureDataset(Dataset):
    def __init__(self, file_paths, labels, is_train=False, seed=None):
        self.file_paths = list(file_paths)
        self.labels = np.array(labels, dtype=np.float32)
        self.is_train = is_train
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        canon = load_canonical(self.file_paths[idx])
        if self.is_train:
            canon = augment_signature(canon, self.rng)
        channels = stages_to_channels(canon)
        x = (channels.astype(np.float32) / 255.0 - MEAN) / STD
        return torch.from_numpy(x), torch.tensor(self.labels[idx])

def gather_data(data_root='data'):
    """Returns (train_paths, test_paths, train_labels, test_labels). Label 0 = genuine, 1 = forged."""
    train_g = sorted(glob.glob(os.path.join(data_root, 'train', 'genuine', '*.*')))
    train_f = sorted(glob.glob(os.path.join(data_root, 'train', 'forged', '*.*')))
    test_g = sorted(glob.glob(os.path.join(data_root, 'test', 'genuine', '*.*')))
    test_f = sorted(glob.glob(os.path.join(data_root, 'test', 'forged', '*.*')))

    if train_g or train_f:
        tr_p = train_g + train_f
        tr_l = [0] * len(train_g) + [1] * len(train_f)
        te_p = test_g + test_f
        te_l = [0] * len(test_g) + [1] * len(test_f)
        return tr_p, te_p, tr_l, te_l

    all_files = sorted(glob.glob(os.path.join(data_root, '**', '*.*'), recursive=True))
    img_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
    paths, labels = [], []
    for f in all_files:
        ext = os.path.splitext(f)[1].lower()
        if ext in img_exts:
            fn = os.path.basename(f).lower()
            parent = os.path.basename(os.path.dirname(f)).lower()
            if 'original' in fn or 'genuine' in fn or 'original' in parent or 'genuine' in parent:
                paths.append(f)
                labels.append(0)
            elif 'forg' in fn or 'forg' in parent:
                paths.append(f)
                labels.append(1)

    if len(paths) > 0:
        return train_test_split(paths, labels, test_size=0.2, random_state=SEED, stratify=labels)
    return [], [], [], []

def make_splits(data_root='data', val_frac=0.15):
    """Train / validation / test. Validation is carved out of the training portion, so the
    test set is never used for choosing the best epoch or the decision threshold."""
    tr_p, te_p, tr_l, te_l = gather_data(data_root)
    if len(tr_p) == 0:
        return None
    tr_p, va_p, tr_l, va_l = train_test_split(tr_p, tr_l, test_size=val_frac,
                                              random_state=SEED, stratify=tr_l)
    return {'train': (tr_p, tr_l), 'val': (va_p, va_l), 'test': (te_p, te_l)}

def save_splits(splits, path=SPLIT_PATH):
    out = {k: {'paths': [str(p) for p in v[0]], 'labels': [int(l) for l in v[1]]} for k, v in splits.items()}
    with open(path, 'w') as f:
        json.dump(out, f, indent=1)

def load_splits(path=SPLIT_PATH):
    with open(path) as f:
        d = json.load(f)
    return {k: (v['paths'], v['labels']) for k, v in d.items()}

def build_model(pretrained=True, dropout=0.5):
    weights = EfficientNet_B2_Weights.DEFAULT if pretrained else None
    m = efficientnet_b2(weights=weights)
    m.classifier[0].p = dropout                                   # stronger dropout (default 0.3)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)   # 1000-class head -> 1 forged logit
    return m

### B2. Load the saved model

The checkpoint contains a threshold selected from the **validation set**. It can differ substantially from 0.50. The live demo reports both decisions so the threshold effect is visible. Do not tune the threshold on the held-out test set.

In [ ]:
if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f'{CKPT_PATH} not found. Run Part A (training) once to create it.')

ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
cfg = ckpt.get('config', {})
model = build_model(pretrained=False, dropout=cfg.get('dropout', 0.5)).to(device)  # no ImageNet download needed
model.load_state_dict(ckpt['model'])
model.eval()

# This checkpoint stores P(Forged) threshold selected on the validation set.
THR = float(ckpt.get('prob_thr', 0.5))
if not 0.0 <= THR <= 1.0:
    raise ValueError(f'Invalid checkpoint probability threshold: {THR}')

saved_classes = cfg.get('class_names')
if saved_classes is not None and list(saved_classes) != CLASS_NAMES:
    raise ValueError(
        f'Checkpoint class mapping {saved_classes} does not match notebook mapping {CLASS_NAMES}. '
        'Do not interpret predictions until the label mapping is reconciled.'
    )

print('Model loaded from      :', CKPT_PATH)
print('Architecture           :', cfg.get('arch', 'efficientnet_b2'))
print('Input size (H x W)     :', cfg.get('img_h', OUT_H), 'x', cfg.get('img_w', OUT_W))
print('Saved at epoch         :', ckpt.get('best_epoch', 'n/a'))
print('Validation AUC (saved) :', round(float(ckpt.get('val_auc', float('nan'))), 4))
print(f'CALIBRATED threshold   : {THR:.4f} ({THR*100:.2f}%)')
print('Decision rule          : P(Forged) >= threshold -> Forged; otherwise Genuine')
print('Class mapping          : 0 = Genuine, 1 = Forged')
print(f'Parameters             : {sum(p.numel() for p in model.parameters())/1e6:.2f} M')
if 'config' not in ckpt:
    print('WARNING: legacy checkpoint has no preprocessing/class metadata. Confirm its training pipeline before trusting predictions.')

### B3. Run the model on the test set

In [ ]:
if os.path.exists(SPLIT_PATH):
    test_paths, test_labels = load_splits()['test']
    print(f'Using the saved test split from {SPLIT_PATH}')
else:
    _, test_paths, _, test_labels = gather_data('data')
    print('WARNING: split.json not found - rebuilt the test set from data/.')
print(f'Test images: {len(test_paths)}')

@torch.no_grad()
def predict_probs(model, paths, labels, batch_size=16):
    model.eval()
    loader = DataLoader(SignatureDataset(paths, labels, is_train=False), batch_size=batch_size, shuffle=False)
    logits, targets = [], []
    for xb, yb in loader:
        logits.append(model(xb.to(device)).squeeze(1).cpu().numpy())
        targets.append(yb.numpy())
    logits = np.concatenate(logits)
    return 1.0 / (1.0 + np.exp(-logits)), np.concatenate(targets).astype(int)

test_probs, test_targets = predict_probs(model, test_paths, test_labels)
test_preds = (test_probs >= THR).astype(int)

pred_df = pd.DataFrame({
    'image_path': test_paths,
    'true_label': test_targets,
    'true_class': [CLASS_NAMES[t] for t in test_targets],
    'forged_prob': test_probs,
    'predicted_label': test_preds,
    'predicted_class': [CLASS_NAMES[p] for p in test_preds],
    'is_correct': test_preds == test_targets,
})
pred_df.to_csv('results/test_predictions.csv', index=False)
print('Saved results/test_predictions.csv')
pred_df.head()

### B4. Evaluation metrics
Positive class = **Forged**.

| Metric | Meaning |
|---|---|
| Accuracy / Balanced Accuracy | overall correctness / mean of the two per-class recalls |
| Precision, Recall (TPR), Specificity (TNR), F1 | standard classification metrics for the Forged class |
| AUC-ROC, Average Precision | threshold-independent ranking quality |
| MCC, Cohen Kappa | chance-corrected agreement (robust summary scores) |
| **FAR** (False Acceptance Rate) | forged signatures wrongly **accepted** as genuine = FN / (FN + TP) |
| **FRR** (False Rejection Rate) | genuine signatures wrongly **rejected** as forged = FP / (FP + TN) |
| **EER** (Equal Error Rate) | the point where FAR = FRR (lower is better) |
| HTER | (FAR + FRR) / 2 at the chosen threshold |

In [ ]:
def compute_metrics(targets, probs, thr):
    preds = (probs >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(targets, preds, labels=[0, 1]).ravel()
    far = fn / (fn + tp) if (fn + tp) else 0.0     # forged accepted as genuine
    frr = fp / (fp + tn) if (fp + tn) else 0.0     # genuine rejected as forged
    fpr, tpr, _ = roc_curve(targets, probs)
    i = int(np.nanargmin(np.abs((1 - tpr) - fpr)))
    eer = float((fpr[i] + (1 - tpr[i])) / 2)
    return {
        'Threshold': thr,
        'Accuracy': accuracy_score(targets, preds),
        'Balanced Accuracy': balanced_accuracy_score(targets, preds),
        'Precision (Forged)': precision_score(targets, preds, zero_division=0),
        'Recall / TPR (Forged)': recall_score(targets, preds, zero_division=0),
        'Specificity / TNR (Genuine)': tn / (tn + fp) if (tn + fp) else 0.0,
        'F1 (Forged)': f1_score(targets, preds, zero_division=0),
        'MCC': matthews_corrcoef(targets, preds),
        'Cohen Kappa': cohen_kappa_score(targets, preds),
        'AUC-ROC': roc_auc_score(targets, probs),
        'Average Precision': average_precision_score(targets, probs),
        'EER': eer,
        'FAR (forged accepted)': far,
        'FRR (genuine rejected)': frr,
        'HTER': (far + frr) / 2,
        'TP (Forged->Forged)': int(tp), 'TN (Genuine->Genuine)': int(tn),
        'FP (Genuine->Forged)': int(fp), 'FN (Forged->Genuine)': int(fn),
    }

metrics_df = pd.DataFrame({
    f'Saved threshold ({THR:.3f})': compute_metrics(test_targets, test_probs, THR),
    'Default threshold (0.500)': compute_metrics(test_targets, test_probs, 0.5),
}).T
metrics_df.to_csv('results/evaluation_metrics.csv')
print('Saved results/evaluation_metrics.csv\n')
display(metrics_df.T.round(4))

print('\nPer-class report (saved threshold):')
print(classification_report(test_targets, test_preds, target_names=CLASS_NAMES, digits=4))

### B5. Evaluation plots

In [ ]:
fpr, tpr, roc_thr = roc_curve(test_targets, test_probs)
prec, rec, _ = precision_recall_curve(test_targets, test_probs)
auc_val = roc_auc_score(test_targets, test_probs)
ap_val = average_precision_score(test_targets, test_probs)
cm = confusion_matrix(test_targets, test_preds, labels=[0, 1])
cm_norm = cm / cm.sum(axis=1, keepdims=True)
m_sel = compute_metrics(test_targets, test_probs, THR)

fig, axes = plt.subplots(2, 3, figsize=(19, 10))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0, 0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0, 0].set_title('Confusion Matrix (counts)', fontweight='bold')
axes[0, 0].set_xlabel('Predicted'); axes[0, 0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', cbar=False, ax=axes[0, 1],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0, 1].set_title('Confusion Matrix (row-normalised)', fontweight='bold')
axes[0, 1].set_xlabel('Predicted'); axes[0, 1].set_ylabel('True')

axes[0, 2].plot(fpr, tpr, color='#7C3AED', lw=2, label=f'ROC (AUC = {auc_val:.3f})')
axes[0, 2].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[0, 2].scatter([m_sel['FRR (genuine rejected)']], [1 - m_sel['FAR (forged accepted)']],
                   color='red', zorder=5, label='Operating point')
axes[0, 2].set_title('ROC Curve', fontweight='bold')
axes[0, 2].set_xlabel('False Positive Rate (= FRR)'); axes[0, 2].set_ylabel('True Positive Rate (= 1 - FAR)')
axes[0, 2].grid(True, linestyle='--', alpha=0.6); axes[0, 2].legend(loc='lower right')

axes[1, 0].plot(rec, prec, color='#EA580C', lw=2, label=f'PR (AP = {ap_val:.3f})')
axes[1, 0].axhline(test_targets.mean(), color='gray', linestyle='--', label='Chance level')
axes[1, 0].set_title('Precision-Recall Curve', fontweight='bold')
axes[1, 0].set_xlabel('Recall'); axes[1, 0].set_ylabel('Precision')
axes[1, 0].grid(True, linestyle='--', alpha=0.6); axes[1, 0].legend(loc='lower left')

axes[1, 1].hist(test_probs[test_targets == 0], bins=25, alpha=0.65, color='#2563EB', label='Genuine')
axes[1, 1].hist(test_probs[test_targets == 1], bins=25, alpha=0.65, color='#DC2626', label='Forged')
axes[1, 1].axvline(THR, color='black', linestyle='--', label=f'Threshold = {THR:.2f}')
axes[1, 1].set_title('Score Distribution', fontweight='bold')
axes[1, 1].set_xlabel('Predicted probability of Forged'); axes[1, 1].set_ylabel('Count')
axes[1, 1].legend()

ths = np.linspace(0, 1, 201)
pos_scores, neg_scores = test_probs[test_targets == 1], test_probs[test_targets == 0]
far_c = np.array([(pos_scores < t).mean() for t in ths])     # forged scored below threshold -> accepted
frr_c = np.array([(neg_scores >= t).mean() for t in ths])    # genuine scored above threshold -> rejected
eer_i = int(np.argmin(np.abs(far_c - frr_c)))
axes[1, 2].plot(ths, far_c, color='#DC2626', label='FAR (forged accepted)')
axes[1, 2].plot(ths, frr_c, color='#2563EB', label='FRR (genuine rejected)')
axes[1, 2].scatter([ths[eer_i]], [(far_c[eer_i] + frr_c[eer_i]) / 2], color='black', zorder=5,
                   label=f'EER = {(far_c[eer_i] + frr_c[eer_i]) / 2:.3f}')
axes[1, 2].axvline(THR, color='gray', linestyle=':')
axes[1, 2].set_title('FAR / FRR vs. Threshold', fontweight='bold')
axes[1, 2].set_xlabel('Threshold'); axes[1, 2].set_ylabel('Error rate')
axes[1, 2].grid(True, linestyle='--', alpha=0.6); axes[1, 2].legend()

plt.suptitle('EfficientNet-B2 Signature Verification - Test Set Evaluation', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('results/evaluation_plots.png', dpi=300)
plt.show()

### B6. Confidence intervals (bootstrap, 1000 resamples of the test set)

In [ ]:
rng = np.random.default_rng(SEED)
n = len(test_targets)
accs, aucs, f1s = [], [], []
for _ in range(1000):
    idx = rng.integers(0, n, n)
    t, p = test_targets[idx], test_probs[idx]
    if len(np.unique(t)) < 2:
        continue
    pr = (p >= THR).astype(int)
    accs.append(accuracy_score(t, pr)); aucs.append(roc_auc_score(t, p)); f1s.append(f1_score(t, pr))

ci_rows = []
for name, vals in [('Accuracy', accs), ('AUC-ROC', aucs), ('F1 (Forged)', f1s)]:
    lo, hi = np.percentile(vals, [2.5, 97.5])
    ci_rows.append({'Metric': name, 'Mean': np.mean(vals), '95% CI low': lo, '95% CI high': hi})
ci_df = pd.DataFrame(ci_rows).round(4)
ci_df.to_csv('results/evaluation_confidence_intervals.csv', index=False)
display(ci_df)

### B7. Most confident mistakes
Preprocessed images (what the network actually sees) of the test signatures that were misclassified with the highest confidence.

In [ ]:
wrong = np.where(test_preds != test_targets)[0]
if len(wrong) == 0:
    print('No misclassified test images.')
else:
    order = wrong[np.argsort(-np.abs(test_probs[wrong] - THR))][:8]
    fig, axes = plt.subplots(2, 4, figsize=(18, 6))
    for ax, i in zip(axes.ravel(), order):
        ax.imshow(load_canonical(test_paths[i]), cmap='gray')
        ax.set_title(f'True: {CLASS_NAMES[test_targets[i]]} -> Pred: {CLASS_NAMES[test_preds[i]]}\nP(forged) = {test_probs[i]:.2f}',
                     fontsize=10, color='#B91C1C')
        ax.axis('off')
    for ax in axes.ravel()[len(order):]:
        ax.axis('off')
    plt.suptitle(f'Misclassified test signatures ({len(wrong)} of {len(test_targets)} in total)', fontweight='bold')
    plt.tight_layout()
    plt.show()

### B8. Live demo - verify a single signature
Shows the exact DIP stages fed to the network and reports the verdict using the checkpoint-calibrated threshold. It also reports the verdict at 0.50 so you can see whether the threshold changes the decision.

**Important:** this is a generic genuine-vs-forged image classifier, not a personalized comparison against an enrolled person's reference signature. A new person's genuine signature can still be misclassified.

In [ ]:
def verify_signature(path, thr=None, show=True, compare_default=True):
    """Classify one image. By default uses the validation-calibrated checkpoint threshold.

    Returns (verdict, forged_probability). This is image-level classification, not
    identity-specific signature matching.
    """
    threshold = THR if thr is None else float(thr)
    if not 0.0 <= threshold <= 1.0:
        raise ValueError(f"Threshold must be between 0 and 1; received {threshold}")

    canon = load_canonical(path)
    ch = stages_to_channels(canon)
    x_np = (ch.astype(np.float32) / 255.0 - MEAN) / STD
    x = torch.from_numpy(x_np).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logit = float(model(x).reshape(-1)[0].item())
        # Numerically stable scalar sigmoid
        prob = float(1.0 / (1.0 + np.exp(-np.clip(logit, -80.0, 80.0))))

    verdict = CLASS_NAMES[int(prob >= threshold)].upper()
    default_verdict = CLASS_NAMES[int(prob >= 0.5)].upper()

    print(f"P(Genuine)                 : {(1.0 - prob):.2%}")
    print(f"P(Forged)                  : {prob:.2%}")
    print(f"Checkpoint calibrated thr. : {threshold:.2%}")
    print(f"Verdict at selected threshold: {verdict}")
    if compare_default and abs(threshold - 0.5) > 1e-8:
        print(f"Verdict at 50% threshold   : {default_verdict}")
        if verdict != default_verdict:
            print("NOTE: The verdict changes because the selected threshold differs from 50%; this does not by itself indicate a model or preprocessing bug.")

    if show:
        raw = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        panels = [('Original', raw), ('Canonical (crop + binarise)', canon),
                  ('Channel 1: median-filtered', ch[0]),
                  ('Channel 2: high-boost', ch[1]),
                  ('Channel 3: Sobel gradient', ch[2])]
        fig, axes = plt.subplots(1, 5, figsize=(20, 3.2))
        for ax, (title, im) in zip(axes, panels):
            ax.imshow(im, cmap='gray')
            ax.set_title(title, fontsize=10)
            ax.axis('off')
        plt.suptitle(
            f'Verdict: {verdict} | P(forged)={prob:.3f} | threshold={threshold:.3f}',
            fontweight='bold'
        )
        plt.tight_layout()
        plt.show()
    return verdict, prob

# Demo on a held-out test image. Replace demo_path with your own image path to test a new image.
if len(test_paths) == 0:
    print('No test images are available. Check data/ and results/split.json.')
else:
    demo_idx = int(np.random.default_rng(SEED).integers(len(test_paths)))
    demo_path = test_paths[demo_idx]  # <- replace with your own image path
    print('Image       :', demo_path)
    print('Ground truth:', CLASS_NAMES[int(test_labels[demo_idx])])
    verdict, prob = verify_signature(demo_path)

---
# Novelty and Modifications of EfficientNet-B2 in this Signature-Forgery Case Study

**What is kept from EfficientNet-B2:** the MBConv + Squeeze-and-Excitation backbone, compound scaling, and the ImageNet-pretrained weights (transfer learning).
**What is adapted:** everything *around and on top of* the backbone, to turn a natural-image classifier into a signature-forgery verifier.

| # | Component | Standard EfficientNet-B2 | This study | Why it helps |
|---|---|---|---|---|
| 1 | **Input representation** | 3 natural RGB colour channels | 3 channels produced by a **DIP pipeline**: Ch1 = contrast-stretched + median-filtered, Ch2 = high-boost (sharpened), Ch3 = Sobel gradient magnitude | Signatures are thin-stroke line art, not colour photos. The channels expose stroke shape, edge sharpness and gradient (direction/pressure-like) cues. Keeping exactly 3 channels means the pretrained stem convolution is reused unchanged. |
| 2 | **Canonicalisation** | Resize + centre crop | Gaussian blur + **Otsu** ink mask, small-blob noise removal, tight bounding-box crop with margin, binarisation, **aspect-preserving letterbox** | Removes pen colour, paper texture, scanner background and position/size variation, so the network judges stroke geometry only. |
| 3 | **Input size / shape** | Square (about 260-288 px) | **192 x 384** (wide) | Signatures are wide; a square resize distorts them or wastes pixels. |
| 4 | **Classification head** | Dropout 0.3 + Linear(1408 -> 1000 classes) | **Dropout 0.5 + Linear(1408 -> 1)** single "forged" logit, BCE-with-logits | Binary verification task; stronger dropout counters over-fitting on a small dataset. |
| 5 | **Augmentation** | Colour jitter, random crops, flips | **Signature-specific**: rotation (+/-6 deg), scale (0.82-1.0), horizontal shear (+/-8 deg), translation, random **erode / dilate** (simulates pen-thickness change), then re-binarise; DIP channels are computed *after* augmentation. No horizontal flip. | Imitates natural intra-writer variation. A mirrored signature is a different signature, so flipping is deliberately avoided. |
| 6 | **Fine-tuning recipe** | Generic | Full fine-tuning, **AdamW** (weight decay 0.05), **head LR = 10 x backbone LR**, cosine annealing, **label smoothing** (targets 0.05 / 0.95) | New head learns quickly while pretrained features change gently; smoothing reduces over-confidence. |
| 7 | **Decision rule** | argmax / fixed 0.5 | **EER-calibrated threshold** selected on a validation split and stored inside the saved checkpoint | Standard in signature verification: the operating point balances accepting forgeries against rejecting genuine signatures. |
| 8 | **Evaluation** | Top-1 accuracy | **FAR, FRR, EER, HTER**, AUC, PR-AUC, MCC, Kappa, confusion matrix, bootstrap CIs | Metrics that matter in forgery detection, not accuracy alone. |
| 9 | **Reproducible protocol** | - | Saved model **and** saved train/val/test split (`results/split.json`); test set never used for model selection | Testing can be repeated later on exactly the same unseen images. |

### Novelty statement
The contribution is **not a new backbone**; it is an **application-specific adaptation**: a *DIP-driven three-channel signature representation* feeding an *ImageNet-pretrained EfficientNet-B2*, trained with *signature-aware augmentation* and a *verification-oriented decision threshold (EER)*. Classical image-processing knowledge (binarisation, sharpening, gradients) is fused with a modern, parameter-efficient CNN, so the pretrained network can be reused on a line-art domain it never saw during ImageNet training.